In [1]:
import yfinance as yf
import pandas as pd
import sqlite3
from datetime import datetime

In [2]:
TICKERS = ['MSFT', 'GOOGL', 'META', 'AMZN', 'NVDA', 'AMD',
           'INTC', 'CRM', 'ORCL', 'ADBE', 'CSCO', 'IBM', 'NOW', 'NFLX']

BENCHMARKS = ['SPY', 'XLK']

START_DATE = '2019-01-01'
END_DATE = '2026-06-25'

ALL_SYMBOLS = TICKERS + BENCHMARKS

In [3]:
data = yf.download(
    ALL_SYMBOLS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True
)

print(f"Shape: {data.shape[0]} rows (trading days) x {data.shape[1]} columns")

[*********************100%***********************]  16 of 16 completed

Shape: 1879 rows (trading days) x 80 columns


In [4]:
data.head()

Price            Close                                               \
Ticker            ADBE        AMD       AMZN         CRM       CSCO   
2019-01-02  224.570007  18.830000  76.956497  133.244797  34.472191   
2019-01-03  215.699997  17.049999  75.014000  128.182388  33.218506   
2019-01-04  226.190002  19.000000  78.769501  135.613815  34.714836   
2019-01-07  229.259995  20.570000  81.475502  139.801376  34.949394   
2019-01-08  232.679993  20.750000  82.829002  143.241837  35.232471   

Price                                                               ...  \
Ticker          GOOGL        IBM       INTC        META       MSFT  ...   
2019-01-02  52.270508  80.143341  40.518047  134.498901  94.193153  ...   
2019-01-03  50.822842  78.543396  38.289032  130.593216  90.727982  ...   
2019-01-04  53.429722  81.611107  40.638535  136.749130  94.947655  ...   
2019-01-07  53.323177  82.188492  40.827862  136.848251  95.068748  ...   
2019-01-08  53.791519  83.357147  41.086067  141.289261  95.758049  ...   

Price        Volume                                                     \
Ticker          IBM      INTC      META      MSFT       NFLX       NOW   
2019-01-02  4434935  18774600  28146200  35329300  116795000  12807500   
2019-01-03  4546648  32267300  22717900  42579100  149696000  12344500   
2019-01-04  4683779  35447300  29002100  44060600  193301000  10978500   
2019-01-07  3923755  22736800  20089300  35656100  186201000   9124500   
2019-01-08  4982726  22749200  26263800  31514400  153592000   8046000   

Price                                                 
Ticker           NVDA      ORCL        SPY       XLK  
2019-01-02  508752000  14320400  126925200  30885800  
2019-01-03  705552000  19868700  144140700  49893400  
2019-01-04  585620000  20984000  142628800  41535600  
2019-01-07  709160000  17967900  103139100  23817200  
2019-01-08  786016000  16255700  102512600  26005200  

[5 rows x 80 columns]

In [5]:
data['Close'].head()

Ticker,ADBE,AMD,AMZN,CRM,CSCO,GOOGL,IBM,INTC,META,MSFT,NFLX,NOW,NVDA,ORCL,SPY,XLK
2019-01-02,224.570007,18.830000,76.956497,133.244797,34.472191,52.270508,80.143341,40.518047,134.498901,94.193153,26.766001,35.664001,3.373052,40.505611,223.805984,28.997625
2019-01-03,215.699997,17.049999,75.014000,128.182388,33.218506,50.822842,78.543396,38.289032,130.593216,90.727982,27.120001,33.824001,3.169262,40.111465,218.465302,27.534184
2019-01-04,226.190002,19.000000,78.769501,135.613815,34.714836,53.429722,81.611107,40.638535,136.749130,94.947655,29.757000,35.846001,3.372309,41.840256,225.783005,28.754494
2019-01-07,229.259995,20.570000,81.475502,139.801376,34.949394,53.323177,82.188492,40.827862,136.848251,95.068748,31.534000,37.334000,3.550842,42.503113,227.563187,29.011646
2019-01-08,232.679993,20.750000,82.829002,143.241837,35.232471,53.791519,83.357147,41.086067,141.289261,95.758049,32.027000,37.622002,3.462442,42.888279,229.701218,29.254778


In [6]:
print("Missing Close prices per ticker:")
print(data['Close'].isna().sum())

Missing Close prices per ticker:
Ticker
ADBE     0
AMD      0
AMZN     0
CRM      0
CSCO     0
GOOGL    0
IBM      0
INTC     0
META     0
MSFT     0
NFLX     0
NOW      0
NVDA     0
ORCL     0
SPY      0
XLK      0
dtype: int64


In [7]:
long_data = data.stack(level='Ticker', future_stack=True).reset_index()

long_data.columns = [c.lower() for c in long_data.columns]
long_data = long_data.rename(columns={'level_0': 'date'})

print(f"Shape: {long_data.shape}")
long_data.head(10)

Shape: (30064, 7)


,date,ticker,close,high,low,open,volume
0,2019-01-02,ADBE,224.570007,226.169998,219.000000,219.910004,2784100
1,2019-01-02,AMD,18.830000,19.000000,17.980000,18.010000,87148700
2,2019-01-02,AMZN,76.956497,77.667999,73.046501,73.260002,159662000
3,2019-01-02,CRM,133.244797,134.503027,130.787313,131.131351,4783900
4,2019-01-02,CSCO,34.472191,34.672844,33.878257,33.934439,23833500
5,2019-01-02,GOOGL,52.270508,52.573323,50.813427,50.908584,31868000
6,2019-01-02,IBM,80.143341,80.678975,77.694733,77.917331,4434935
7,2019-01-02,INTC,40.518047,40.853689,39.390632,39.554149,18774600
8,2019-01-02,META,134.498901,136.312973,127.440886,127.867150,28146200
9,2019-01-02,MSFT,94.193153,94.779995,92.162486,92.730701,35329300


In [8]:
import sqlite3

DB_PATH = "../data/market.db"

conn = sqlite3.connect(DB_PATH)

long_data.to_sql('prices', conn, if_exists='replace', index=False)

result = conn.execute("SELECT COUNT(*) FROM prices").fetchone()
print(f"Saved {result[0]} rows to the 'prices' table in {DB_PATH}")

conn.close()
print("Database connection closed.")

Saved 30064 rows to the 'prices' table in ../data/market.db
Database connection closed.


In [9]:
import time

earnings_frames = []

for ticker in TICKERS:
    try:
        t = yf.Ticker(ticker)
        ed = t.get_earnings_dates(limit=60)
        if ed is not None and len(ed) > 0:
            ed = ed.reset_index()
            ed['ticker'] = ticker
            earnings_frames.append(ed)
            print(f"{ticker}: {len(ed)} earnings dates")
        else:
            print(f"{ticker}: no earnings dates returned")
    except Exception as e:
        print(f"{ticker}: ERROR - {e}")
    time.sleep(1)

print(f"\nCollected earnings data for {len(earnings_frames)} companies")

MSFT: 100 earnings dates
GOOGL: 88 earnings dates
META: 57 earnings dates
AMZN: 100 earnings dates
NVDA: 100 earnings dates
AMD: 100 earnings dates
INTC: 100 earnings dates
CRM: 89 earnings dates
ORCL: 100 earnings dates
ADBE: 100 earnings dates
CSCO: 100 earnings dates
IBM: 100 earnings dates
NOW: 57 earnings dates
NFLX: 97 earnings dates

Collected earnings data for 14 companies


In [10]:
earnings_df = pd.concat(earnings_frames, ignore_index=True)

earnings_df.columns = [c.lower().replace(' ', '_') for c in earnings_df.columns]

print(f"Total rows: {len(earnings_df)}")
print(f"Columns: {earnings_df.columns.tolist()}")
earnings_df.head(10)

Total rows: 1288
Columns: ['earnings_date', 'eps_estimate', 'reported_eps', 'surprise(%)', 'ticker']


,earnings_date,eps_estimate,reported_eps,surprise(%),ticker
0,2026-07-29 16:00:00-04:00,4.24,NaN,NaN,MSFT
1,2026-04-29 16:00:00-04:00,4.06,4.27,5.22,MSFT
2,2026-01-28 16:00:00-05:00,3.92,4.14,5.69,MSFT
3,2025-10-29 16:00:00-04:00,3.66,4.13,12.73,MSFT
4,2025-07-30 16:00:00-04:00,3.38,3.65,8.09,MSFT
5,2025-04-30 16:00:00-04:00,3.22,3.46,7.40,MSFT
6,2025-01-29 16:00:00-05:00,3.12,3.23,3.47,MSFT
7,2024-10-30 16:00:00-04:00,3.11,3.30,6.13,MSFT
8,2024-07-30 16:00:00-04:00,2.94,2.95,0.23,MSFT
9,2024-04-25 16:00:00-04:00,2.84,2.94,3.40,MSFT


In [11]:
earnings_df = earnings_df.rename(columns={'surprise(%)': 'eps_surprise_pct'})
earnings_df['earnings_date'] = pd.to_datetime(earnings_df['earnings_date'], utc=True).dt.tz_localize(None).dt.normalize()
earnings_df = earnings_df[earnings_df['earnings_date'] >= '2019-01-01']
print(f"Rows after filtering to 2019+: {len(earnings_df)}")

Rows after filtering to 2019+: 434


In [12]:
conn = sqlite3.connect("../data/market.db")
earnings_df.to_sql('earnings', conn, if_exists='replace', index=False)
count = conn.execute("SELECT COUNT(*) FROM earnings").fetchone()[0]
print(f"Saved {count} rows to 'earnings' table")
conn.close()

Saved 434 rows to 'earnings' table


In [13]:
# ============================================================================
# VIX pull — used as a market-regime feature (handles COVID-era volatility).
# We keep VIX separately (not in `prices`) because it's a vol index, not a stock.
# ============================================================================
import yfinance as yf

vix_raw = yf.download('^VIX', start=START_DATE, end=END_DATE,
                      auto_adjust=True, progress=False)
vix_df = vix_raw['Close'].reset_index()
vix_df.columns = ['date', 'vix_close']
vix_df['date'] = pd.to_datetime(vix_df['date'])
print(f"Pulled {len(vix_df)} VIX days "
      f"from {vix_df['date'].min().date()} to {vix_df['date'].max().date()}")
vix_df.head()

Pulled 1880 VIX days from 2019-01-02 to 2026-06-24


,date,vix_close
0,2019-01-02,23.219999
1,2019-01-03,25.450001
2,2019-01-04,21.379999
3,2019-01-07,21.400000
4,2019-01-08,20.469999


In [14]:
# ============================================================================
# Recompute returns:
#   * abnormal returns benchmarked to **XLK** (sector-adjusted), not SPY
#   * adds `vix_close` on/before the earnings date (risk-regime feature)
#   * adds `is_covid` boolean for the 2020-02-20 to 2021-06-30 window
#
# Self-contained: reloads prices + earnings from SQLite and defines the
# `get_return` helper inline, so this cell works even if upstream variables
# were reset (e.g., fresh kernel session).
# ============================================================================
import sqlite3
import yfinance as yf
from datetime import datetime

START_DATE = '2019-01-01'
END_DATE   = '2026-06-25'

# COVID window: Feb 19-20 2020 ≈ pre-COVID US equity peak (S&P 500 closed at
# record high on Feb 19 2020); Jun 30 2021 ≈ end of the acute-recovery
# volatility regime. Earnings dates inside this band get `is_covid=True`,
# and we summarize on-/off-window stats separately as a robustness check.
COVID_START = pd.Timestamp('2020-02-20')
COVID_END   = pd.Timestamp('2021-06-30')

# ---- 1. Reload data from disk (self-contained) ----------------------------
conn = sqlite3.connect('../data/market.db')
prices_long = pd.read_sql('SELECT * FROM prices', conn, parse_dates=['date'])
earnings_df = pd.read_sql('SELECT * FROM earnings', conn, parse_dates=['earnings_date'])
conn.close()
print(f'Reloaded {len(prices_long):,} price rows, {len(earnings_df)} earnings events')

# ---- 2. Re-pull VIX if kernel was reset (defensive) ----------------------
# `vix_df` is normally set by the previous `vix_pull_001` cell. If the user
# restarted the kernel and only ran this cell, we re-pull made it here.
if 'vix_df' not in globals():
    print('vix_df is not in memory — re-pulling ^VIX')
    _vix_raw = yf.download('^VIX', start=START_DATE, end=END_DATE,
                           auto_adjust=True, progress=False)
    vix_df = _vix_raw['Close'].reset_index()
    vix_df.columns = ['date', 'vix_close']
    vix_df['date'] = pd.to_datetime(vix_df['date'])

# ---- 3. Pivot to wide for fast single-ticker lookups ---------------------
prices_wide = prices_long.pivot(index='date', columns='ticker',
                                values='close').sort_index()
vix_by_date = vix_df.set_index('date')['vix_close'].sort_index()

# ---- 4. Helper function ---------------------------------------------------
# CRITICAL: this function is the only one that uses `get_return` downstream.
# It is defined AT MODULE LEVEL (zero indent). If you ever see an indentation
# error here, that's the bug — the function must be at column 0 so it's
# visible to the for-loop below.
def get_return(ticker, dd, n_days):
    """Forward n-day return for ticker starting at the first trading day
    on or after `dd`. Returns None if the window runs off the data."""
    future = prices_wide.index[prices_wide.index >= dd]
    if len(future) == 0:
        return None
    start_idx = future[0]
    start_pos = prices_wide.index.get_loc(start_idx)
    end_pos = start_pos + n_days
    if end_pos >= len(prices_wide.index):
        return None
    end_idx = prices_wide.index[end_pos]
    if ticker not in prices_wide.columns:
        return None
    px_s = prices_wide.loc[start_idx, ticker]
    px_e = prices_wide.loc[end_idx, ticker]
    if pd.isna(px_s) or pd.isna(px_e):
        return None
    return float(px_e / px_s - 1)

# ---- 5. Per-event returns + XLK-adjusted abnormal + VIX + COVID flag -----
rows = []
for _, row in earnings_df.iterrows():
    if pd.isna(row['reported_eps']):
        continue  # skip rows where reported EPS is missing (future dates)
    date = row['earnings_date']
    ticker = row['ticker']
    r1, r30, r90 = (get_return(ticker, date, n) for n in (1, 30, 90))
    xlk1, xlk30, xlk90 = (get_return('XLK', date, n) for n in (1, 30, 90))
    # VIX close on or before the earnings date — captures pre-earnings risk
    # sentiment, not post-news reaction
    vix_prior = vix_by_date[vix_by_date.index <= date]
    vix_val = float(vix_prior.iloc[-1]) if len(vix_prior) > 0 else None
    rows.append({
        'ticker': ticker,
        'earnings_date': date,
        'return_1d':    r1,
        'return_30d':   r30,
        'return_90d':   r90,
        # NOTE: benchmark is XLK (tech sector), not SPY — strips out the
        # common tech-sector beta shared by all 15 names
        'abnormal_1d':  r1  - xlk1  if r1  is not None and xlk1  is not None else None,
        'abnormal_30d': r30 - xlk30 if r30 is not None and xlk30 is not None else None,
        'abnormal_90d': r90 - xlk90 if r90 is not None and xlk90 is not None else None,
        'vix_close': vix_val,
        'is_covid':  bool(COVID_START <= date <= COVID_END),
    })

returns_df = pd.DataFrame(rows)
print(f'Computed XLK-adjusted returns for {len(returns_df)} earnings events '
      f'({returns_df["is_covid"].sum()} in COVID window, '
      f'{returns_df["vix_close"].notna().sum()} with VIX matched)')
returns_df.head(10)


Reloaded 30,064 price rows, 434 earnings events
Computed XLK-adjusted returns for 420 earnings events (73 in COVID window, 420 with VIX matched)


,ticker,earnings_date,return_1d,return_30d,return_90d,abnormal_1d,abnormal_30d,abnormal_90d,vix_close,is_covid
0,MSFT,2026-04-29,-0.039297,-0.078393,NaN,-0.041748,-0.229860,NaN,18.809999,False
1,MSFT,2026-01-28,-0.099931,-0.163721,-0.141304,-0.084117,-0.087396,-0.377041,16.350000,False
2,MSFT,2025-10-29,-0.029157,-0.105577,-0.249259,-0.016860,-0.078648,-0.174147,16.920000,False
3,MSFT,2025-07-30,0.039475,-0.022224,-0.055296,0.046804,-0.046329,-0.164472,15.480000,False
4,MSFT,2025-04-30,0.076254,0.213756,0.265359,0.061585,0.058495,-0.000440,24.700001,False
5,MSFT,2025-01-29,-0.061809,-0.141977,0.072881,-0.063664,-0.038067,0.037451,16.559999,False
6,MSFT,2024-10-30,-0.060527,0.041453,-0.120780,-0.028430,0.004783,-0.026665,20.350000,False
7,MSFT,2024-07-30,-0.010806,0.002087,0.050566,-0.053157,-0.034262,-0.094405,17.690001,False
8,MSFT,2024-04-25,0.018244,0.064090,0.028408,0.006958,-0.027003,-0.034536,15.370000,False
9,MSFT,2024-01-30,-0.026946,0.017812,0.041141,-0.005948,-0.015964,-0.028372,13.310000,False


In [15]:
# ============================================================================
# Persist the corrected returns to SQLite (overwrites the old SPY-based table)
# ============================================================================
conn = sqlite3.connect("../data/market.db")
returns_df.to_sql('returns', conn, if_exists='replace', index=False)
saved = conn.execute("SELECT COUNT(*) FROM returns").fetchone()[0]
cols  = conn.execute("PRAGMA table_info(returns)").fetchall()
conn.close()
print(f"Saved {saved} rows to 'returns' table")
print("Columns:", [c[1] for c in cols])

Saved 420 rows to 'returns' table
Columns: ['ticker', 'earnings_date', 'return_1d', 'return_30d', 'return_90d', 'abnormal_1d', 'abnormal_30d', 'abnormal_90d', 'vix_close', 'is_covid']


In [16]:
# ============================================================================
# Sanity checks — quick read of the corrected returns distribution
# ============================================================================
print("=== Abnormal 1d (XLK-adjusted) ===")
print(returns_df['abnormal_1d'].describe().round(4))

print("\n=== Abnormal 30d (XLK-adjusted) ===")
print(returns_df['abnormal_30d'].describe().round(4))

print("\n=== Abnormal 90d (XLK-adjusted) ===")
print(returns_df['abnormal_90d'].describe().round(4))

print("\n=== VIX distribution (joined to earnings dates) ===")
print(returns_df['vix_close'].describe().round(2))

print("\n=== COVID-window cohort (vs non-COVID) ===")
print(returns_df.groupby('is_covid')['abnormal_1d'].agg(['count','mean','std']).round(4))

print("\n=== By ticker: mean abnormal 1d ===")
print(returns_df.groupby('ticker')['abnormal_1d']
      .agg(['count','mean','std']).round(4).sort_values('mean', ascending=False))

=== Abnormal 1d (XLK-adjusted) ===
count    420.0000
mean      -0.0010
std        0.0781
min       -0.3502
25%       -0.0533
50%       -0.0054
75%        0.0505
max        0.3414
Name: abnormal_1d, dtype: float64

=== Abnormal 30d (XLK-adjusted) ===
count    415.0000
mean      -0.0059
std        0.1157
min       -0.4585
25%       -0.0767
50%       -0.0151
75%        0.0684
max        0.3281
Name: abnormal_30d, dtype: float64

=== Abnormal 90d (XLK-adjusted) ===
count    402.0000
mean      -0.0012
std        0.2120
min       -0.5537
25%       -0.1343
50%       -0.0246
75%        0.1027
max        0.8107
Name: abnormal_90d, dtype: float64

=== VIX distribution (joined to earnings dates) ===
count    420.00
mean      20.22
std        7.29
min       11.94
25%       15.61
50%       18.05
75%       23.18
max       75.47
Name: vix_close, dtype: float64

=== COVID-window cohort (vs non-COVID) ===
          count    mean     std
is_covid                       
False       347  0.0000  0.0805
Tr

In [17]:
# ============================================================================
# Robustness check — same summary, but excluding 2020 Q1 -> 2021 Q2
# (the COVID-vol regime). Confirm results aren't COVID-driven.
# ============================================================================
robust_df = returns_df[~returns_df['is_covid']]
print(f"Excluding COVID window: {len(robust_df)} events "
      f"({len(returns_df) - len(robust_df)} dropped)")
print()
print(robust_df['abnormal_1d'].describe().round(4))
print()
print(robust_df.groupby('ticker')['abnormal_1d']
      .agg(['count','mean','std']).round(4).sort_values('mean', ascending=False))

Excluding COVID window: 347 events (73 dropped)

count    347.0000
mean       0.0000
std        0.0805
min       -0.3502
25%       -0.0534
50%       -0.0040
75%        0.0537
max        0.3414
Name: abnormal_1d, dtype: float64

        count    mean     std
ticker                       
NOW        25  0.0242  0.0791
NVDA       25  0.0227  0.0648
ORCL       24  0.0213  0.1065
META       25  0.0090  0.1150
GOOGL      25  0.0044  0.0571
IBM        25  0.0017  0.0705
CRM        24  0.0003  0.0687
MSFT       25  0.0003  0.0395
CSCO       25 -0.0010  0.0643
AMZN       25 -0.0016  0.0725
AMD        25 -0.0036  0.0794
NFLX       25 -0.0208  0.1093
ADBE       24 -0.0267  0.0636
INTC       25 -0.0304  0.0962
